In [ ]:
root="../data"

In [ ]:
from torchvision.datasets import OxfordIIITPet
import matplotlib.pyplot as plt

train = OxfordIIITPet(root="../data", split="trainval", target_types="category", download=True)
test = OxfordIIITPet(root="../data", split="test", target_types="category", download=True)

print(len(train), len(test))

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, (img, label) in zip(axes.flat, [train[i] for i in range(8)]):
    ax.imshow(img)
    ax.set_title(train.classes[label])
    ax.axis("off")
plt.show()

In [ ]:
from collections import Counter
import numpy as np

labels = np.array(train._labels)
counts = Counter(labels)

for cls_id, n in sorted(counts.items(), key=lambda x: x[1]):
    print(f"{train.classes[cls_id]:30s} {n}")

print("\nmin:", min(counts.values()), "max:", max(counts.values()), "classes:", len(counts))

In [ ]:
from PIL import Image

sizes = [Image.open(p).size for p in train._images]  # opening a file only reads the header, so this is fast
widths, heights = zip(*sizes)
aspect = np.array(widths) / np.array(heights)

print("width  min/median/max:", min(widths), int(np.median(widths)), max(widths))
print("height min/median/max:", min(heights), int(np.median(heights)), max(heights))
print("aspect min/median/max:", aspect.min().round(2), np.median(aspect).round(2), aspect.max().round(2))

plt.hist(aspect, bins=50); plt.xlabel("width / height"); plt.title("Aspect ratios"); plt.show()

In [ ]:
def show_class(name, n=6):
    cls_id = train.classes.index(name)
    idxs = np.where(labels == cls_id)[0][:n]
    fig, axes = plt.subplots(1, n, figsize=(2.5 * n, 3))
    for ax, i in zip(axes, idxs):
        ax.imshow(train[i][0]); ax.axis("off")
    fig.suptitle(name); plt.show()

for name in ["American Pit Bull Terrier", "Staffordshire Bull Terrier", "British Shorthair", "Russian Blue"]:
    show_class(name)

In [ ]:
from sklearn.model_selection import train_test_split

all_idx = np.arange(len(train))
train_ix, val_ix = train_test_split(all_idx, test_size = 0.2, stratify = labels, random_state = 42)

print(len(train_ix), len(val_ix)) 

print(np.bincount(labels[train_ix])[:5], np.bincount(labels[val_ix])[:5])

np.save("../data/train_ix.npy", val_ix)
np.save("../data/val_ix.npy", val_ix)

In [ ]:
img, label = train[33]
print(type(img), img.size, img.mode)   # a PIL image: (width, height), "RGB"
print(label, train.classes[label])     # an int, and the breed name it stands for
img                                    # a bare PIL image as the last line of a cell shows it inline

In [ ]:
import torch
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),  # random zoom/crop
    transforms.RandomHorizontalFlip(),                    # a mirrored dog is still the same breed
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [ ]:
from torch.utils.data import Subset, DataLoader
from torchvision.datasets import OxfordIIITPet

train_idx = np.load("../data/train_ix.npy")
val_idx   = np.load("../data/val_ix.npy")

# same images on disk, two different transforms
base_train = OxfordIIITPet(root="../data", split="trainval", target_types="category", transform=train_tf)
base_eval  = OxfordIIITPet(root="../data", split="trainval", target_types="category", transform=eval_tf)

train_ds = Subset(base_train, train_idx)
val_ds   = Subset(base_eval,  val_idx)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=0)

xb, yb = next(iter(train_loader))
print(xb.shape, yb.shape, yb[:10])

In [ ]:
peek_loader = DataLoader(train_ds, batch_size=32, shuffle=False)
xb, yb = next(iter(peek_loader))

In [ ]:
def denorm(x):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (x * std + mean).clamp(0, 1)

fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for ax, x, y in zip(axes.flat, xb, yb):
    ax.imshow(denorm(x).permute(1, 2, 0))
    ax.set_title(train.classes[y], fontsize=8); ax.axis("off")
plt.show()

In [ ]:
import torch.nn as nn
from torchvision import models

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
print(model.fc)   # the original last layer: 512 features -> 1000 ImageNet classes

for p in model.parameters():
    p.requires_grad = False             # freeze everything...

model.fc = nn.Linear(model.fc.in_features, 37)   # ...then replace the head (new layers are trainable by default)
model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"trainable params: {trainable:,} / {total:,}")

In [ ]:
import time

def train_one_epoch(model, loader, loss_fn, optimizer):
    model.eval()   # see note below
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)                 # [B, 37] raw scores
        loss = loss_fn(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n

@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        total_loss += loss_fn(logits, y).item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

for epoch in range(5):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader, loss_fn, optimizer)
    va_loss, va_acc = evaluate(model, val_loader, loss_fn)
    print(f"epoch {epoch+1}  train loss {tr_loss:.3f} acc {tr_acc:.3f}  |  "
          f"val loss {va_loss:.3f} acc {va_acc:.3f}  ({time.time()-t0:.0f}s)")

torch.save(model.state_dict(), "../data/resnet18_linear_probe.pt")